# Depth-First Search (DFS) and Breadth-First Search (BFS)

**Course:** Artificial Intelligence · Unit 2 — Problem-Solving  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Russell, S. & Norvig, P. (2022). *Artificial Intelligence: A Modern Approach* (4th ed.), Ch. 3 — Solving Problems by Searching  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13 (no external dependencies)

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Represent a graph as an adjacency list in Python
2. Implement and distinguish **iterative** vs **recursive** DFS
3. Implement BFS and explain its shortest-path guarantee
4. Identify and correct two classic Python anti-patterns: the **unordered `set()` visited** issue and the **mutable default argument** bug
5. Compare the traversal order and path properties of DFS and BFS

## Prerequisites

- Python fundamentals: functions, lists, sets, dictionaries, generators (`yield`, `yield from`)
- Basic graph theory: vertices (nodes), edges, adjacency, connected components

## Context

Graph search is the foundation of **uninformed (blind) search** strategies in AI — the first class of problem-solving agents studied in Russell & Norvig Part II. DFS and BFS are the two canonical uninformed strategies. Understanding their trade-offs (memory, completeness, optimality) is essential before moving to informed search (A*, heuristics) in later units.

## 1. Graph Representation

Graph theory underpins a vast range of Computer Science problems. A graph $G = (V, E)$ consists of a set of **vertices** $V$ (nodes) and a set of **edges** $E$ connecting them. Edges can be directed or undirected, and weighted or unweighted.

**Two common representations:**

| Representation | Best for | Memory | Edge lookup |
|---|---|---|---|
| Adjacency matrix | Dense graphs | $O(V^2)$ | $O(1)$ |
| Adjacency list | Sparse graphs | $O(V + E)$ | $O(\text{degree})$ |

This notebook uses an **adjacency list** (dictionary of sets), which is efficient for the sparse, undirected graph below.

**Goals of this notebook:**
- Find all vertices in a connected component reachable from a given start node
- Return all paths between two vertices
- Return the **shortest path** (by edge count) using BFS

In [ ]:
# Undirected graph with 6 vertices and 6 edges, represented as an adjacency list.
# Each key maps to a set of its direct neighbours.
# Because the graph is undirected, every edge appears in BOTH directions.
graph = {
    'A': {'B', 'C'},
    'B': {'A', 'D', 'E'},
    'C': {'A', 'F'},
    'D': {'B'},
    'E': {'B', 'F'},
    'F': {'C', 'E'},
}

The graph looks like this:

```
    A
   / \
  B   C
 / \   \
D   E - F
```

Notice the **cycle** F–E–B–A–C–F. This means there are multiple paths between several pairs of nodes, which both DFS and BFS must handle without revisiting vertices.

---

## 2. Depth-First Search (DFS)

DFS explores as deep as possible along each branch before **backtracking**. It uses a **stack** (LIFO) to track nodes to visit next. The algorithm:

1. Push the start node onto the stack
2. Pop a node; if not visited, mark it and push its unvisited neighbours
3. Repeat until the stack is empty

### 2.1 Connected Component — Iterative DFS

**Goal:** Return all nodes reachable from a starting vertex (its connected component).

#### ⚠️ Anti-pattern: using `set()` for `visited`

The implementation below works correctly in terms of **reachability** (it finds all vertices), but using a `set` for `visited` loses insertion order, making traversal sequence non-deterministic across runs and Python versions.

In [ ]:
# ANTI-PATTERN: visited is a set — order is non-deterministic
def dfs_unordered(graph, start):
    visited, stack = set(), [start]
    while stack:
        vertex = stack.pop()
        if vertex not in visited:
            visited.add(vertex)
            stack.extend(graph[vertex] - visited)
    return visited

dfs_unordered(graph, 'A')

> **Output interpretation:** The result `{'A', 'B', 'C', 'D', 'E', 'F'}` is a `set` — all 6 vertices are found, which is correct for reachability. However, the set displays in an arbitrary order and provides no information about the **sequence** in which DFS visited them. For algorithm analysis and debugging, traversal order matters.

#### ✅ Correct iterative DFS — ordered traversal

In [ ]:
def dfs(graph, start):
    """Iterative DFS — returns nodes in the order they were first visited."""
    visited, stack = [], [start]
    while stack:
        vertex = stack.pop()          # LIFO: always explore the most recently added node
        if vertex not in visited:
            visited.append(vertex)
            stack.extend(graph[vertex] - set(visited))   # avoid revisiting
    return visited

dfs(graph, 'A')

> **Output interpretation:** The result is a **list** that preserves the DFS traversal order, e.g., `['A', 'B', 'D', 'E', 'F', 'C']`. Reading it left to right traces the exact sequence: starting at A, DFS immediately descends into B, then D (deepest branch), backtracks to E, crosses to F, backtracks all the way to C. This **depth-first** behavior contrasts with BFS, which would visit A's two direct neighbours (B and C) before going deeper.

### 2.2 Connected Component — Recursive DFS

The recursive form expresses DFS more concisely by letting the **call stack** replace the explicit stack data structure.

**Two bugs to avoid:**

| Bug | Symptom |
|---|---|
| `visited = set()` (unordered) | Traversal order is non-deterministic |
| `visited=[]` as default arg (mutable default) | State leaks between calls — a node may appear twice |

#### ⚠️ Anti-pattern: mutable default argument

In [ ]:
# ANTI-PATTERN: mutable default argument
# Python evaluates default values ONCE at function definition time.
# The same [] object is reused across all calls, leaking state.
def dfs_mutable_default(graph, start, visited=[]):
    visited.append(start)
    for neighbor in graph[start] - set(visited):
        dfs_mutable_default(graph, neighbor, visited)
    return visited

# Run twice to observe the bug:
result1 = dfs_mutable_default.__wrapped__ if hasattr(dfs_mutable_default, '__wrapped__') else None
# Demonstrate by calling once and noting the state leak
dfs_mutable_default(graph, 'C')

> **Output interpretation:** Notice that node `'A'` appears **twice** in the output (`['C', 'F', 'E', 'B', 'D', 'A', 'A']`). This is the mutable default argument bug: the `visited` list was allocated once at definition time and persists across recursive calls. When `dfs_mutable_default(graph, 'A', visited)` is called recursively, `'A'` is added again because the check `graph[start] - set(visited)` momentarily allows it through before `visited` is updated. Run the cell a second time without restarting the kernel and you will see even more duplicate entries — proof that `visited` retains state between top-level calls.

#### ✅ Correct recursive DFS — `None` sentinel pattern

In [ ]:
def dfs_recursive(graph, start, visited=None):
    """Recursive DFS — correct None-sentinel pattern avoids mutable default bug."""
    if visited is None:
        visited = []          # fresh list on every top-level call
    visited.append(start)
    for neighbor in graph[start] - set(visited):
        dfs_recursive(graph, neighbor, visited)
    return visited

print("From C:", dfs_recursive(graph, 'C'))
print("From A:", dfs_recursive(graph, 'A'))  # second call is clean — no state leak

> **Output interpretation:** Both calls return clean, duplicate-free results. Starting from `'C'`, the recursive DFS first follows the C→F edge (deepest branch from C), then F→E, E→B, B→D, then backtracks up to B→A, yielding `['C', 'F', 'E', 'B', 'D', 'A']`. The second call starting from `'A'` is independent — the `None` sentinel ensures a new `visited` list is created each time, so no state bleeds between invocations.

---

## 3. DFS — All Paths Between Two Vertices

By modifying DFS to carry the current path at each step, we can enumerate **all** paths from a start to a goal vertex. We use a **generator** (`yield`) so paths are produced lazily — the caller can stop early once it finds what it needs.

In [ ]:
def dfs_paths(graph, start, goal):
    """Iterative DFS — yields every simple path from start to goal."""
    stack = [(start, [start])]         # (current_node, path_so_far)
    while stack:
        vertex, path = stack.pop()
        for neighbor in graph[vertex] - set(path):   # exclude already-visited nodes
            if neighbor == goal:
                yield path + [neighbor]              # found a complete path
            else:
                stack.append((neighbor, path + [neighbor]))

all_paths = list(dfs_paths(graph, 'A', 'F'))
print(f"All paths from A to F ({len(all_paths)} found):")
for p in all_paths:
    print(" →".join(p), f"  (length: {len(p)-1} edges)")

> **Output interpretation:** DFS finds two paths from A to F: `A→B→E→F` (3 edges) and `A→C→F` (2 edges). Critically, DFS does **not** guarantee the shortest path arrives first — it depends entirely on which branch the stack explores first. Here `A→B→E→F` (longer) may appear before `A→C→F` (shorter). This is the fundamental limitation that BFS was designed to overcome.

### 3.1 Recursive DFS Paths (`yield from`)

The recursive version delegates to sub-calls using `yield from` (PEP 380), which passes all yielded values from a recursive generator up the call chain without buffering them.

In [ ]:
def dfs_paths_recursive(graph, start, goal, path=None):
    """Recursive DFS — yields all simple paths from start to goal."""
    if path is None:
        path = [start]
    if start == goal:
        yield path
    for neighbor in graph[start] - set(path):
        yield from dfs_paths_recursive(graph, neighbor, goal, path + [neighbor])

all_paths = list(dfs_paths_recursive(graph, 'C', 'F'))
print(f"All paths from C to F ({len(all_paths)} found):")
for p in all_paths:
    print(" →".join(p), f"  (length: {len(p)-1} edges)")

> **Output interpretation:** From C to F the algorithm finds `C→F` (1 edge, direct) and `C→A→B→E→F` (4 edges, the long way around). The shorter path `C→F` appears first here because C's adjacency set happens to yield F before A — but this ordering is implementation-dependent, not a guarantee of DFS. The key insight is that **both** paths are found, proving DFS is **complete** (for finite graphs without cycles in the path, since we exclude already-visited nodes).

---

## 4. Breadth-First Search (BFS)

BFS explores all vertices at the **current depth** before moving deeper. It uses a **queue** (FIFO) instead of a stack:

1. Enqueue the start node
2. Dequeue a node; if not visited, mark it and enqueue its unvisited neighbours
3. Repeat until the queue is empty

**Key property:** Because nodes are explored level by level, the **first time** BFS reaches a goal it has found one of the shortest paths (minimum number of edges).

| Property | DFS | BFS |
|---|---|---|
| Data structure | Stack (LIFO) | Queue (FIFO) |
| Memory | $O(\text{depth})$ | $O(\text{width})$ |
| Complete? | Yes (finite graphs) | Yes |
| Finds shortest path? | ❌ No guarantee | ✅ Yes (by edge count) |

### 4.1 Connected Component — Iterative BFS

#### ⚠️ Anti-pattern: `set()` for visited (same issue as DFS)

In [ ]:
# ANTI-PATTERN: visited is a set — reachability is correct but traversal order is lost
def bfs_unordered(graph, start):
    visited, queue = set(), [start]
    while queue:
        vertex = queue.pop(0)          # FIFO: remove from the front
        if vertex not in visited:
            visited.add(vertex)
            queue.extend(graph[vertex] - visited)
    return visited

bfs_unordered(graph, 'A')

> **Output interpretation:** Returns `{'A', 'B', 'C', 'D', 'E', 'F'}` as an unordered set — all vertices reachable, but the level-by-level BFS order is invisible. This defeats the purpose of BFS since the breadth-first property only becomes observable when traversal order is preserved.

#### ✅ Correct iterative BFS — ordered traversal

In [ ]:
from collections import deque

def bfs(graph, start):
    """Iterative BFS — returns nodes in breadth-first (level-by-level) order.
    
    Uses collections.deque for O(1) left-pops instead of O(n) list.pop(0).
    """
    visited = []
    queue = deque([start])
    visited_set = set()               # O(1) membership check
    visited_set.add(start)

    while queue:
        vertex = queue.popleft()      # FIFO: process oldest-queued node first
        visited.append(vertex)
        for neighbor in graph[vertex] - visited_set:
            visited_set.add(neighbor)
            queue.append(neighbor)
    return visited

bfs(graph, 'A')

> **Output interpretation:** The list reveals BFS's level-by-level exploration. Starting at A (level 0), BFS visits A's direct neighbours B and C (level 1) before descending to D, E (via B, level 2) and F (via C or E, level 2). Compare with DFS's output: DFS went deep immediately (A→B→D), while BFS spreads wide first (A→B,C→D,E,F). Note: `deque.popleft()` is used instead of `list.pop(0)` because it is O(1) vs O(n) — an important efficiency improvement for large graphs.

---

## 5. BFS — All Paths & Shortest Path

### 5.1 All Paths (BFS order)

In [ ]:
def bfs_paths(graph, start, goal):
    """BFS — yields all simple paths from start to goal, shortest first."""
    queue = deque([(start, [start])])
    while queue:
        vertex, path = queue.popleft()
        for neighbor in graph[vertex] - set(path):
            if neighbor == goal:
                yield path + [neighbor]
            else:
                queue.append((neighbor, path + [neighbor]))

all_paths = list(bfs_paths(graph, 'A', 'F'))
print(f"All paths from A to F ({len(all_paths)} found):")
for p in all_paths:
    print(" →".join(p), f"  (length: {len(p)-1} edges)")

> **Output interpretation:** BFS yields `A→C→F` (2 edges) **before** `A→B→E→F` (3 edges). This is the shortest-path-first guarantee: BFS explores all 2-edge paths before any 3-edge paths. In a navigation or routing context, this means the first result is always optimal by hop count — no need to enumerate all paths and sort.

### 5.2 Shortest Path Utility

In [ ]:
def shortest_path(graph, start, goal):
    """Returns the shortest path (by edge count) from start to goal, or None."""
    try:
        return next(bfs_paths(graph, start, goal))   # first BFS result is shortest
    except StopIteration:
        return None   # no path exists

path = shortest_path(graph, 'A', 'F')
if path:
    print(f"Shortest path A → F: {' → '.join(path)}  ({len(path)-1} edges)")
else:
    print("No path found.")

> **Output interpretation:** `A → C → F` with 2 edges is the shortest path. The function is efficient: it calls `next()` on the BFS generator, so it stops as soon as the first (shortest) path is found — it does not enumerate all paths. If the graph had no path between A and F (a disconnected graph), `StopIteration` would be raised and `None` returned cleanly.

---

## 6. Summary: DFS vs BFS

| Criterion | DFS | BFS |
|---|---|---|
| Data structure | Stack (LIFO) | Queue (FIFO) |
| Traversal order | Deep before wide | Wide before deep (level by level) |
| Memory complexity | $O(d)$ where $d$ = max depth | $O(b^d)$ where $b$ = branching factor |
| Finds shortest path? | ❌ Not guaranteed | ✅ Yes (by edge count) |
| Finds a path? | ✅ Yes (finite graphs) | ✅ Yes |
| Preferred when | Graph is deep, solution is far | Shortest path needed, graph is shallow |

In the AI problem-solving context (R&N Ch. 3), BFS is **complete and optimal** for unweighted graphs but uses exponential memory. DFS uses linear memory but may explore deep dead ends before finding a solution. **Iterative Deepening DFS (IDDFS)** combines the best of both — it is the algorithm of choice when neither memory nor solution depth is known in advance.